In [ ]:
# ==============================================================================
# INSTALASI DEPENDENSI (Jalankan sel ini HANYA jika pustaka belum terpasang)
# Hilangkan tanda pagar (#) pada baris di bawah untuk memasang dependensi:
# ==============================================================================
# !pip install numpy pandas matplotlib scipy scikit-learn geopandas rasterio quantile-forest


# Uji Lingkungan Kerja Pelatihan RS & DSM

Pelatihan Pemanfaatan Remote Sensing dan Digital Soil Mapping, PS2DL UGM, 3–5 November 2026.

Notebook ini memastikan Google Colab di akun Anda siap dipakai untuk praktik. Jalankan sel satu per satu dari atas ke bawah dengan menekan tombol ▶ di kiri setiap sel atau **Shift + Enter**.

Setelah semua sel selesai, **ambil tangkapan layar hasil sel terakhir** dan kirimkan ke panitia.

## 1. Memasang pustaka

Sel ini memasang pustaka yang belum tersedia di Colab. Prosesnya memerlukan sekitar 1–2 menit.

In [ ]:
# !pip install -q rasterio geopandas quantile-forest

## 2. Memeriksa pustaka

In [ ]:
import importlib

pustaka = ["numpy", "pandas", "matplotlib", "scipy", "sklearn",
           "geopandas", "rasterio", "quantile_forest"]
gagal = []
for nama in pustaka:
    try:
        modul = importlib.import_module(nama)
        print(f"OK     {nama:16s} {getattr(modul, '__version__', '-')}")
    except Exception as e:
        gagal.append(nama)
        print(f"GAGAL  {nama:16s} {e}")
print()
print("Semua pustaka siap." if not gagal else f"Pustaka bermasalah: {', '.join(gagal)}")

## 3. Menyambungkan Google Drive

Colab akan meminta izin mengakses Google Drive Anda. Pilih akun yang sama, lalu izinkan.

Folder data pelatihan (`pelatihan_dsm`) dibagikan oleh panitia. Agar terbaca di sini, buka tautan folder tersebut di Google Drive, lalu pilih **Tambahkan pintasan ke Drive → Drive Saya**.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

folder = Path("/content/drive/MyDrive/pelatihan_dsm")
drive_ok = folder.exists()
if drive_ok:
    print("Folder data ditemukan. Isinya:")
    for p in sorted(folder.iterdir()):
        print("  -", p.name)
else:
    print("Drive tersambung, tetapi folder 'pelatihan_dsm' belum ditemukan di Drive Saya.")
    print("Tambahkan pintasan folder dari tautan panitia, lalu jalankan sel ini lagi.")
    print("(Jika paket data belum dibagikan, lanjutkan saja ke sel berikutnya.)")

## 4. Uji fungsi

Sel ini membuat data acak kecil untuk menguji bahwa raster dapat ditulis dan dibaca, serta model *random forest* dan *quantile regression forest* dapat dilatih. Data ini **bukan data tanah** dan hasilnya tidak bermakna. Yang diuji hanya apakah prosesnya berjalan.

In [ ]:
import numpy as np
import rasterio
from rasterio.io import MemoryFile
from rasterio.transform import from_origin
from sklearn.ensemble import RandomForestRegressor
from quantile_forest import RandomForestQuantileRegressor

rng = np.random.default_rng(0)

# Raster acak 50 x 50 piksel, ditulis dan dibaca kembali di memori
larik = rng.random((50, 50)).astype("float32")
profil = dict(driver="GTiff", height=50, width=50, count=1, dtype="float32",
              crs="EPSG:32749", transform=from_origin(0, 50, 1, 1))
with MemoryFile() as mem:
    with mem.open(**profil) as dst:
        dst.write(larik, 1)
    with mem.open() as src:
        raster_ok = np.allclose(src.read(1), larik)
print("Tulis-baca raster :", "OK" if raster_ok else "GAGAL")

# Model pada data acak
X = rng.random((200, 4))
y = X[:, 0] * 2 + rng.normal(0, 0.1, 200)
rf = RandomForestRegressor(n_estimators=50, random_state=0).fit(X, y)
qrf = RandomForestQuantileRegressor(n_estimators=50, random_state=0).fit(X, y)
q = qrf.predict(X[:5], quantiles=[0.05, 0.5, 0.95])
model_ok = rf.predict(X[:5]).shape == (5,) and q.shape == (5, 3)
print("Random forest     :", "OK" if model_ok else "GAGAL")
print("Quantile RF       :", "OK" if model_ok else "GAGAL")

## 5. Ringkasan

Ambil tangkapan layar hasil sel di bawah ini dan kirimkan ke panitia.

In [ ]:
import datetime, platform
print("Waktu uji     :", datetime.datetime.now().strftime("%Y-%m-%d %H:%M"))
print("Python        :", platform.python_version())
print("Pustaka       :", "OK" if not gagal else "BERMASALAH -> " + ", ".join(gagal))
print("Google Drive  :", "folder data ditemukan" if drive_ok else "tersambung, folder data belum ditemukan")
print("Uji fungsi    :", "OK" if (raster_ok and model_ok) else "BERMASALAH")